# 🫀 실험 19 — **동작점을 2단계로**: 유도는 천장에 닿았고 남은 병목은 유병률이다

**MedKOS / `notebooks/exp19_two_stage_operating_point.ipynb`** · 퀘스트 `ailab-2026-0015`
**다운로드 없음** · 실험16~18 arm 재사용 · 새로 학습 **30회 (~50분)**

---

## 실험18 이 남긴 것 — 좋은 소식과 벽

**좋은 소식**: `{I,II,V2,V5}`(5전극) 최악 D **+0.0076** · 경보배수 **1.03**.
7부위 전부 12유도의 0.008 AUROC 이내다. **유도 문제는 사실상 풀렸다.**

**벽**: 민감도 0.90 동작점을 PPV 로 번역하면 —

| 부위 | 유병률 | PPV(5전극) | **PPV(12유도)** | 경보 1건당 위양성 |
|---|---|---|---|---|
| ASMI | 10.8% | 45.7% | 48.4% | 1건 |
| IMI | 12.3% | 35.3% | 34.5% | 2건 |
| ILMI | 2.2% | 10.8% | 12.2% | 8건 |
| ALMI | 1.3% | 6.4% | 5.9% | 15건 |
| AMI | 1.6% | 3.2% | 3.3% | 30건 |
| LMI | 0.9% | 1.8% | 1.8% | 55건 |
| IPLMI | 0.23% | 0.4% | 0.6% | **251건** |

**12유도도 똑같다.** 유도를 아무리 최적화해도 이 열은 안 바뀐다.

> **유도 최적화는 천장에 닿았다. 남은 병목은 유도가 아니라 유병률 × 동작점 설계다.**

## 무엇이 잘못 설계돼 있나 — 7부위 각각에 민감도 0.90

지금은 **부위마다 독립적으로** 민감도 0.90 을 건다. 그런데 임상 순서는 그렇지 않다:

| 단계 | 질문 | 놓치면 | 그래서 |
|---|---|---|---|
| **1** | 이 사람 **MI 인가** | **사람이 죽는다** | 민감도 최우선 (0.95) |
| **2** | **어느 벽**인가 | 이미 ①에서 잡혔다 | **특이도 쪽으로 옮겨도 된다** |

지금 설계는 ②에도 ①과 같은 안전 마진을 걸어 **놓쳐도 되는 것에 값을 치른다.**
`IPLMI` 경보율 52% 가 그 대가다 — 전체 ECG 의 절반을 '하후측벽 의심' 으로 띄운다.

**2단계로 나누면 분모가 바뀐다**: '전체 ECG' → 'MI 양성으로 걸린 ECG'.
유병률이 10배 가까이 오르므로 **같은 모델·같은 성능으로 PPV 가 따라 오른다.**
저장된 arm 으로 **학습 0회** 계산된다.

## 실험18 에서 철회할 것 두 개

**① A1 라우팅 결론.** 라우팅 쪽만 시드 3개 **예측을 평균**하고 기준은 **시드별 AUROC 의
평균**이었다 — 앙상블과 라우팅이 섞였다. 7부위 중 **5부위는 모든 겹이 같은 구성을
골랐는데도**(라우팅이 한 일이 없다) D 가 좋아졌다. **양쪽에 똑같이 앙상블을 걸고 다시 잰다.**

**② 그 과정에서 나온 진짜 발견은 남는다.** 시드 3개 예측 평균만으로 AUROC
**+0.0256**(중앙값, 범위 +0.013~+0.067) — 이 퀘스트가 쫓아온 **어떤 유도 효과보다 크다.**
이것도 양쪽 공정 비교로 다시 확인한다.

## 무엇을 얼마나 돌리나

| | 학습 |
|---|---|
| `{I,II,V2,V5}` 시드 3·4·5 (승자 확정용 6시드) | **15** |
| **MI-any 전용 이진 헤드** 시드 0·1·2 (1단계 게이트용) | **15** |
| **B0** 앙상블 이득 — 전 구성 공정 비교 | 0 |
| **B1** 라우팅 재측정 — 양쪽 앙상블 | 0 |
| **B2** 2단계 동작점 | 0 |
| **합** | **30회 (~50분)** |

## 2단계 동작점을 어떻게 정의하나 (사전 고정)

- **1단계 점수** `s₁` — 게이트 후보 **세 가지를 다 재고** 희소 4부위 민감도를 지키면서
  경보를 가장 많이 줄이는 것을 쓴다:
  · `any` — 7부위 확률의 **최댓값**에 MI-any 민감도 0.95
  · `safe` — 같은 점수인데 **모든 부위**가 0.95 를 넘도록(부위별 분위수의 최솟값)
  · **`head`** — **MI-any 전용 이진 헤드를 따로 학습**해 그 점수로 게이트 (15회)
  임계값은 **검증 겹에서 잡아 테스트 겹에 적용**(교차적합 — 실험14 규약).

> ⚠️ **왜 전용 헤드가 필요한가.** 픽스처에서 `any` 게이트는 희소 부위 양성례를
> **21~24% 버렸고**(통과율 0.76~0.89), `safe` 게이트는 민감도를 지키는 대신
> **아무것도 못 걸렀다**(통과율 1.00, 경보 변화 −0.002). 원인은 하나다 —
> '부위 확률의 최댓값' 은 **흔한 부위가 임계값을 정한다.**
> 게이트는 게이트용으로 학습해야 한다.
- **2단계**: `s₁ ≥ 임계값` 인 ECG 만 대상. 부위별 임계값은
  **최종 민감도가 0.855 가 되도록** 검증 겹에서 잡는다(테스트 겹에 적용).
- **보고**: 두 단계를 통과한 최종 경보의 **PPV·경보율**을 단일단계와 나란히 놓는다.
- ⚠️ **두 단계는 독립이 아니다 — `0.95 × 0.90` 으로 계산하면 틀린다.**
  1단계의 민감도 0.95 는 **MI-any 기준**이고, 특정 부위 양성례가 그 게이트를 통과할
  확률은 부위마다 다르다(어려운 부위일수록 낮다). 픽스처에서 실측 **0.721 vs 단일 0.825**
  로 벌어졌다. 그래서 2단계 임계값을 '0.90' 이 아니라 **최종 민감도를 맞추도록** 잡고,
  게이트에서 탈락한 양성례는 **놓친 것으로 센다.** 그래야 두 설계의 민감도가 설계상
  같아지고 **경보율 비교가 성립한다.**
- 단일단계도 같은 **0.855** 로 잡는다. 노트북이 실측 민감도를 찍어 전제를 확인한다.

## 사전등록 (결과 보기 전에 고정)

판정은 시드 t-CI. 점추정 채점 금지.

| | 예측 | 성격 |
|---|---|---|
| **G0** | 유도 순서 · arm 재사용 정합성(드리프트 ≤ 0.02) | 전제 |
| **P-1 ★★** | **6시드**에서 `{I,II,V2,V5}` 의 D 가 7부위 전부 **< 0.02** | 5전극 사양 확정. 실험18 최악이 +0.0076 이라 0.05 는 너무 헐겁다 |
| **P-2 ★★** | 2단계 동작점이 **같은 최종 민감도**에서 단일단계보다 **경보율을 낮춘다** — 희소 4부위(ILMI·ALMI·AMI·LMI) 중앙값 | 동작점 재설계의 값어치 |
| **P-3 ★** | 2단계에서 희소 4부위 **PPV 중앙값이 2배 이상** | 배포 가능성 |
| **P-4** | **앙상블 이득**: 전 구성·전 부위에서 (시드평균 예측 AUROC) − (시드별 AUROC 평균) > 0 | 실험18 발견의 공정 재확인 |
| **P-5** | **라우팅 이득**: 양쪽 앙상블 후에도 부위별 라우팅이 최선 고정 구성보다 최악 D 를 낮춘다 | 실험18 A1 재측정. ❌ 면 실험20(구조 변경) 불필요 |

> **P-2·P-3 이 이 실험의 본체다.** P-1 은 확정 절차, P-4·P-5 는 실험18 정정이다.

## 이 실험이 무엇을 정하나

- `P-2`·`P-3` ✅ → **배포 사양이 완성된다**: 5전극 + 2단계 동작점. 외부검증으로 간다.
- `P-2` ❌ → 2단계로도 안 되면 남은 길은 **유병률이 높은 코호트**(응급실 흉통 =
  MIMIC-IV-ECG 급성)뿐이다. 그쪽이 논문의 주 코호트가 된다.
- `P-5` ❌ → 실험20(평면 분리 헤드)은 **만들 필요 없다.** 착수 근거가 사라진다.


In [ ]:
# CELL 0 — 공용 사전점검
class LabelVocabError(ValueError):
    pass

# ── pipelines/ecg_preflight.py 인라인 (원본·테스트는 repo)
def assert_label_vocab(requested, available, kind="label", counts=None, min_count=1):
    """요청한 이름이 실제 어휘에 **전부** 있는지 확인한다. 하나라도 없으면 예외.

    0건은 "데이터에 그 소견이 없다"가 아니라 **대개 이름을 잘못 골랐다**는 뜻이다.
    그 둘을 구별하려고 어휘 자체를 대조한다.

    requested : 쓰려는 이름들
    available : 데이터에서 실제로 관측된 이름 집합
    counts    : {이름: 건수} (있으면 min_count 미만도 함께 보고)
    """
    requested, available = list(requested), set(available)
    unknown = [r for r in requested if r not in available]
    if unknown:
        raise LabelVocabError(
            f"{kind} 어휘에 없는 이름 {unknown}.\n"
            f"  → 0건이 나온 이유는 '데이터에 없어서'가 아니라 **이름이 틀려서**다.\n"
            f"  실제 어휘({len(available)}개): {sorted(available)}"
        )
    thin = []
    if counts:
        thin = [(r, counts.get(r, 0)) for r in requested if counts.get(r, 0) < min_count]
    return {"ok": True, "n_requested": len(requested), "thin": thin}

def decide(lo, hi, thr, direction):
    """사전등록 관문의 **유일한** 계약: 지지(True) / 기각(False) / 미결(None).

    CI 가 임계값을 걸치면 **기각이 아니라 미결**이다. 검정력 부족을 반증으로
    위장하지 않기 위해서다. 점추정 2분 채점은 금지한다(실험13b·14 에서 그 실수를 했다).
    """
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if direction == ">":
        if lo > thr:
            return True
        if hi < thr:
            return False
    else:
        if hi < thr:
            return True
        if lo > thr:
            return False
    return None

MARK = {True: "✅ 지지", False: "❌ 기각", None: "⚠️ 미결"}

def assert_arm_shape(arm, expected_rows, name="arm"):
    """저장된 arm 의 행 수가 **겹 크기**인지 확인한다.

    MedKOSRun.save_arm 은 그 겹의 예측만 저장한다(전체 길이가 아니다).
    겹 순서는 `np.where(CV == k)[0]` 의 오름차순이므로,
      OOF[np.where(CV == k)[0]] = load_arm(...)      ← 이렇게 **넣는다**
      load_arm(...)[전역인덱스]                       ← 이렇게 자르면 IndexError
    실험15d G0 에서 이 혼동으로 터졌다.
    """
    n = arm.shape[0]
    if n != expected_rows:
        raise ValueError(
            f"{name} 행 수 {n} != 기대 {expected_rows}.\n"
            "  → arm 은 **겹 크기**로 저장된다. 전역 인덱스로 자르지 말고 "
            "OOF[np.where(CV==k)[0]] = arm 형태로 넣을 것."
        )
    return {"ok": True, "rows": n}

FRONTAL_IDENTITIES = (
    ("III", 2, lambda I, II: II - I),                 # 아인트호벤
    ("aVR", 3, lambda I, II: -(I + II) / 2.0),        # 골드버거
    ("aVL", 4, lambda I, II: I - II / 2.0),
    ("aVF", 5, lambda I, II: II - I / 2.0),
)

def assert_lead_order(X, tol=0.02, sample=200, seed=0):
    """12유도 캐시의 **채널 순서**를 신호 자체로 검증한다.

    헤더의 유도 이름을 믿지 말고 아인트호벤·골드버거 항등식으로 확인한다:
        III = II − I,  aVR = −(I+II)/2,  aVL = I − II/2,  aVF = II − I/2
    넷이 모두 맞으면 0..5 = I,II,III,aVR,aVL,aVF 이고 표준 순서상 6..11 = V1..V6 이다.
    → `{I,II}` = [0,1], `{II,V1}` = [1,6] 을 쓸 근거가 생긴다.

    유도 순서를 틀리면 **예외 없이 조용히 다른 실험**이 된다. 그래서 잰다.
    ※ 원신호(mV) 전제 — 채널별로 정규화한 배열에는 쓸 수 없다.
    """
    import numpy as np
    if X.ndim != 3 or X.shape[2] != 12:
        raise ValueError(f"X 는 (n, t, 12) 여야 한다 — 받은 모양 {X.shape}")
    rs = np.random.RandomState(seed)
    idx = rs.choice(len(X), size=min(sample, len(X)), replace=False)
    S = X[idx].astype("float64")
    I, II = S[:, :, 0], S[:, :, 1]
    report, bad = {}, []
    for name, j, f in FRONTAL_IDENTITIES:
        want = f(I, II)
        scale = np.abs(want).mean() + 1e-9
        err = float(np.abs(S[:, :, j] - want).mean() / scale)
        report[name] = err
        if err > tol:
            bad.append(f"{name}(ch{j}) 상대오차 {err:.3f}")
    if bad:
        raise ValueError(
            "유도 순서가 표준(I,II,III,aVR,aVL,aVF,V1..V6)이 아니다: " + ", ".join(bad) + "\n"
            "  → 항등식이 깨졌다는 것은 채널 배치가 다르거나 채널별 정규화가 걸렸다는 뜻이다.\n"
            "  마스크 인덱스([0,1] 사지 · [1,6] II+V1)를 그대로 쓰면 조용히 다른 실험이 된다."
        )
    return {"ok": True, "n_checked": len(idx), "rel_err": report}

print("사전점검 적재: assert_label_vocab · decide · assert_arm_shape · assert_lead_order")

In [ ]:
# CELL 1 — 설정 + 실험16~18 arm 연결
!pip -q install wfdb

import os, sys, json, time, ast, numpy as np
try:
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = "/content/drive/MyDrive"
except Exception as e:
    print("⚠️ Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

# ★★ 실험15~18 과 한 글자도 달라선 안 되는 블록
K_FOLD, EPOCHS, SEED0, NMIN = 5, 20, 20260801, 50
SITE_CANDIDATES = ["IMI", "ILMI", "IPMI", "IPLMI", "ASMI", "AMI", "ALMI", "LMI", "PMI"]
SITE_PLANE = {"IMI": "전두면", "ILMI": "혼합", "IPMI": "혼합", "IPLMI": "혼합",
              "ASMI": "횡단면", "AMI": "횡단면", "ALMI": "혼합",
              "LMI": "혼합", "PMI": "횡단면"}
# ★★ 여기까지

# 0=I 1=II 2=III 3=aVR 4=aVL 5=aVF 6=V1 7=V2 8=V3 9=V4 10=V5 11=V6
LEADS = {"I+II": [0, 1], "I+II+V1": [0, 1, 6], "I+II+V2": [0, 1, 7],
         "I+II+V1+V5": [0, 1, 6, 10], "I+II+V2+V5": [0, 1, 7, 10],
         "12": list(range(12))}
N_ELEC = {"I+II": 3, "I+II+V1": 4, "I+II+V2": 4,
          "I+II+V1+V5": 5, "I+II+V2+V5": 5, "12": 10}

WIN, REF = "I+II+V2+V5", "12"          # 실험18 의 승자
SEEDS_OLD, SEEDS_NEW = [0, 1, 2], [3, 4, 5]
SEEDS_ALL = SEEDS_OLD + SEEDS_NEW
WANT = {WIN: SEEDS_ALL, REF: SEEDS_ALL, "I+II+V1": SEEDS_ALL,
        "I+II": SEEDS_OLD, "I+II+V2": SEEDS_OLD, "I+II+V1+V5": SEEDS_OLD}

ANY_HEAD   = True       # ★ MI-any 전용 이진 헤드를 따로 학습한다 (15회 추가)
#   왜: 1단계 점수로 '부위 확률의 최댓값' 을 쓰면 **흔한 부위가 임계값을 정해서**
#   희소·어려운 부위 양성례를 먼저 버린다(픽스처에서 통과율 0.76~0.89 로 떨어졌다).
#   반대로 모든 부위를 지키도록 임계값을 낮추면 게이트가 아무것도 못 거른다.
#   → 게이트는 게이트용으로 따로 학습하는 것이 맞다.
D_THR      = 0.02       # P-1 — 실험18 최악이 +0.0076 이라 0.05 는 너무 헐겁다
SENS_L1    = 0.95       # 1단계 MI 유무
SENS_L2    = 0.90       # 2단계 부위 (1단계 통과 집단 안에서)
SENS_FINAL = SENS_L1 * SENS_L2          # ≈ 0.855 — 단일단계도 이 값으로 맞춰 비교
SCARCE     = ["ILMI", "ALMI", "AMI", "LMI"]   # ★ 사전지정: 희소하지만 n>=200 인 4부위
PPV_MULT   = 2.0        # P-3 — PPV 중앙값 2배
DRIFT_THR, ON_DRIFT = 0.02, "retrain"

CONFIG = dict(exp="exp19_two_stage_operating_point", quest="ailab-2026-0015",
              parent_exp=["exp16_four_cfg", "exp17_wearable", "exp18_confirm"],
              purpose=("유도 최적화는 천장에 닿았다(5전극 최악 D +0.0076). "
                       "남은 병목인 유병률×동작점을 2단계 설계로 공략한다"),
              change_one_thing="실험15~18 과 백본·분할·에폭·손실·시드공식 동일. 동작점만 재설계",
              retracting={"실험18 A1": "라우팅 쪽만 시드 예측을 평균해 앙상블과 교란 — 양쪽 앙상블로 재측정",
                          "실험18 A1 풀": "V2 구성이 후보에 없었다 — 이번엔 포함"},
              leads=LEADS, n_electrodes=N_ELEC, winner=WIN, seeds=SEEDS_ALL,
              two_stage=dict(sens_l1=SENS_L1, sens_l2=SENS_L2, sens_final=SENS_FINAL,
                             l1_score="7부위 확률의 최댓값", l1_label="MI-any",
                             threshold="검증 겹에서 잡아 테스트 겹에 적용(교차적합)"),
              scarce=SCARCE, any_head=ANY_HEAD,
              judged_on="시드 수준 t-CI — 점추정 채점 금지",
              predictions={
                  "P-1": f"6시드에서 {WIN} 의 D 가 7부위 전부 < {D_THR}",
                  "P-2": f"같은 최종 민감도 {SENS_FINAL:.3f} 에서 2단계가 희소 4부위 경보율 중앙값을 낮춘다",
                  "P-3": f"2단계에서 희소 4부위 PPV 중앙값이 {PPV_MULT} 배 이상",
                  "P-4": "앙상블 이득 > 0 (전 구성·전 부위, 양쪽 공정 비교)",
                  "P-5": "양쪽 앙상블 후에도 부위별 라우팅이 최선 고정 구성보다 최악 D 를 낮춘다"},
              caveat=("2단계는 민감도가 곱해진다(0.95x0.90=0.855). 단일단계도 0.855 로 "
                      "다시 잡아 비교한다 — 같은 조건이 아니면 비교가 성립하지 않는다"),
              k_fold=K_FOLD, epochs=EPOCHS, seed0=SEED0, nmin=NMIN)
np.random.seed(SEED0)
try:
    import matplotlib, matplotlib.font_manager as fm, subprocess
    if not any("Nanum" in f.name for f in fm.fontManager.ttflist):
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"], capture_output=True)
        fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    matplotlib.rc("font", family="NanumGothic")
    matplotlib.rcParams["axes.unicode_minus"] = False
except Exception as e:
    print("한글 폰트 설정 생략:", e)

run = MedKOSRun("exp19_two_stage", CONFIG, project=PROJECT)

REG = os.path.join(PROJECT, "registry.jsonl")
DIRS = {}
for line in (open(REG) if os.path.exists(REG) else []):
    try:
        r = json.loads(line)
    except Exception:
        continue
    if r.get("exp_id") in ("exp16_four_cfg", "exp17_wearable", "exp18_confirm") \
            and os.path.isdir(r.get("dir", "")):
        DIRS[r["exp_id"]] = r["dir"]
if not DIRS:
    raise RuntimeError("registry.jsonl 에서 실험16~18 을 하나도 못 찾았습니다")
PARENTS = [DIRS[k] for k in ("exp18_confirm", "exp17_wearable", "exp16_four_cfg")
           if k in DIRS]
run.log("부모 실행:")
for k in ("exp18_confirm", "exp17_wearable", "exp16_four_cfg"):
    run.log(f"  {k}: {DIRS.get(k, '없음')}")

def arm_at(d, name):
    p = os.path.join(d, "arms", name, "probs.npy")
    return np.load(p) if os.path.exists(p) else None

def find_arm(c, sd, k):
    a = run.load_arm(f"{c}_s{sd}_f{k}")
    if a is not None:
        return a
    for d in PARENTS:
        a = arm_at(d, f"{c}_s{sd}_f{k}")
        if a is not None:
            return a
    return None

def has_parent_arm(c, sd, k):
    return any(os.path.exists(os.path.join(d, "arms", f"{c}_s{sd}_f{k}", "probs.npy"))
               for d in PARENTS)

In [ ]:
# CELL 2 — 캐시 + 라벨 + 【G0】
import pandas as pd, subprocess

PTB = "/content/ptbxl"; os.makedirs(PTB, exist_ok=True)
d = os.path.join(PTB, "ptbxl_database.csv")
if not (os.path.exists(d) and os.path.getsize(d) > 0):
    subprocess.run(["wget", "-q", "-O", d,
                    "https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv"])
df = pd.read_csv(d, index_col="ecg_id")

CACHE = run.data("ptbxl_12lead_all.npz")
if not os.path.exists(CACHE):
    raise RuntimeError(f"전량 캐시가 없습니다: {CACHE}")
z = np.load(CACHE, allow_pickle=True)
X, FOLD10, EID = z["X"], z["fold"], z["eid"]
CV = (FOLD10 - 1) % K_FOLD

lead_chk = assert_lead_order(X)
run.log("【G0-a】 유도 순서 ✅ " + " · ".join(f"{k} {v:.4f}"
                                          for k, v in lead_chk["rel_err"].items()))

dfa = df.loc[EID]
dfa["codes"] = dfa.scp_codes.apply(lambda s: sorted(ast.literal_eval(s).keys()))
vocab = {c for cs in dfa.codes for c in cs}
counts = {s: int(sum(s in cs for cs in dfa.codes)) for s in SITE_CANDIDATES}
assert_label_vocab(SITE_CANDIDATES, vocab, kind="scp 코드", counts=counts, min_count=NMIN)
SITES = [s for s in SITE_CANDIDATES if counts[s] >= NMIN]
NS = len(SITES)
S18 = json.load(open(os.path.join(DIRS["exp18_confirm"], "result.json"),
                     encoding="utf-8"))["sites"]
if SITES != S18:
    raise RuntimeError(f"부위 목록/순서가 실험18과 다릅니다: {SITES} vs {S18}")
Ymul = np.stack([[s in c for s in SITES] for c in dfa.codes]).astype("float32")
YANY = (Ymul.sum(1) > 0).astype(bool)          # ★ 1단계 라벨: 7부위 중 하나라도
run.log(f"【G0-b】 부위 정합 ✅ {NS}개 · MI-any {int(YANY.sum()):,}건 "
        f"({YANY.mean():.3%})")
for s in SCARCE:
    if s not in SITES:
        raise RuntimeError(f"희소 지정 부위 {s} 가 SITES 에 없습니다")

MASKS = {c: np.zeros(12, "float32") for c in LEADS}
for c, idx in LEADS.items():
    MASKS[c][idx] = 1.0

In [ ]:
# CELL 3 — 학습 15회 (승자 6시드 확정) · 나머지는 재사용
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import roc_auc_score as _auc

def build_head(seed):
    tf.keras.utils.set_random_seed(seed)
    si = layers.Input((X.shape[1], 12))
    x = si
    for f, k in ((32, 9), (64, 7), (128, 5), (128, 3)):
        x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling1D(2)(x)
    h = layers.Dense(64, activation="relu")(layers.GlobalAveragePooling1D()(x))
    h = layers.Dropout(0.3)(h); h = layers.Dense(64, activation="relu")(h)
    m = models.Model(si, layers.Dense(NS, activation="sigmoid")(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
              loss="binary_crossentropy")
    return m

def split(k):
    te = np.where(CV == k)[0]; rest = np.where(CV != k)[0]
    rs_ = np.random.RandomState(SEED0 + k); rs_.shuffle(rest)
    n_val = max(int(len(rest) * 0.12), 200)
    return te, rest[:n_val], rest[n_val:]

def train_one(c, sd, k):
    te, va, tr = split(k)
    mk = MASKS[c]
    m = build_head(SEED0 + 100 * k + 15 + sd)
    m.fit(X[tr] * mk, Ymul[tr], validation_data=(X[va] * mk, Ymul[va]),
          epochs=EPOCHS, batch_size=128, verbose=0)
    p = m.predict(X[te] * mk, batch_size=512, verbose=0)
    tf.keras.backend.clear_session()
    return te, p

# 【G0-c】 재사용 정합성
run.log("\n【G0-c】 재사용 정합성 — {I,II} 시드0 재학습 대조")
_ref = np.zeros((len(EID), NS), "float32")
for k in range(K_FOLD):
    a = find_arm("I+II", 0, k)
    if a is None:
        raise RuntimeError(f"I+II_s0_f{k} 없음")
    _ref[np.where(CV == k)[0]] = a
_chk = np.zeros((len(EID), NS), "float32"); _t = time.time()
for k in range(K_FOLD):
    te, p = train_one("I+II", 0, k); _chk[te] = p
PER = (time.time() - _t) / K_FOLD
DRIFT_MAX = max(abs(_auc(Ymul[:, j].astype(bool), _chk[:, j])
                    - _auc(Ymul[:, j].astype(bool), _ref[:, j]))
                for j in range(NS))
REUSE_OK = DRIFT_MAX <= DRIFT_THR
run.log(f"  최대 |ΔAUROC| = {DRIFT_MAX:.4f} · 1회 {PER:.0f}s → "
        + ("✅ 재사용 확정" if REUSE_OK else f"⚠️ {ON_DRIFT}"))
if not REUSE_OK and ON_DRIFT == "stop":
    raise RuntimeError(f"드리프트 {DRIFT_MAX:.4f}")

NO_REUSE = (not REUSE_OK) and ON_DRIFT == "retrain"
TODO = {(c, sd, k) for c, _s in WANT.items() for sd in _s for k in range(K_FOLD)
        if NO_REUSE or not has_parent_arm(c, sd, k)}
run.log(f"\n학습 대상 {len(TODO)}회 (예상 ≈ {len(TODO) * PER / 60:.0f}분)")
# P[c][sd] = (전체 × 부위) OOF 확률
P = {c: {} for c in WANT}
t0, done = time.time(), 0
for c, _sds in WANT.items():
    for sd in _sds:
        P[c][sd] = np.zeros((len(EID), NS), "float32")
        for k in range(K_FOLD):
            te = np.where(CV == k)[0]
            a = run.load_arm(f"{c}_s{sd}_f{k}")
            if a is None and (c, sd, k) not in TODO:
                a = find_arm(c, sd, k)
            if a is None:
                te, a = train_one(c, sd, k)
                run.save_arm(f"{c}_s{sd}_f{k}", a); done += 1
            assert_arm_shape(a, len(te), name=f"{c}_s{sd}_f{k}")
            P[c][sd][te] = a
        run.log(f"  {c:<12} 시드{sd} 준비 ({done} 학습 · {time.time()-t0:.0f}s)")
# ── MI-any 전용 이진 헤드 (1단계 게이트용)
PANY = {}
if ANY_HEAD:
    def build_any(seed):
        """★ 백본 동일, 출력만 1개. 라벨은 YANY."""
        tf.keras.utils.set_random_seed(seed)
        si = layers.Input((X.shape[1], 12))
        x = si
        for f, k in ((32, 9), (64, 7), (128, 5), (128, 3)):
            x = layers.Conv1D(f, k, padding="same", activation="relu")(x)
            x = layers.BatchNormalization()(x)
            x = layers.MaxPooling1D(2)(x)
        h = layers.Dense(64, activation="relu")(layers.GlobalAveragePooling1D()(x))
        h = layers.Dropout(0.3)(h); h = layers.Dense(64, activation="relu")(h)
        m = models.Model(si, layers.Dense(1, activation="sigmoid")(h))
        m.compile(optimizer=tf.keras.optimizers.Adam(1e-3, clipnorm=1.0),
                  loss="binary_crossentropy")
        return m

    Yany = YANY.astype("float32")[:, None]
    run.log(f"\nMI-any 전용 헤드 학습 ({len(SEEDS_OLD) * K_FOLD}회)")
    for sd in SEEDS_OLD:
        PANY[sd] = np.zeros(len(EID), "float32")
        for k in range(K_FOLD):
            te = np.where(CV == k)[0]
            a = run.load_arm(f"MIANY_{WIN}_s{sd}_f{k}")
            if a is None:
                for d in PARENTS:
                    a = arm_at(d, f"MIANY_{WIN}_s{sd}_f{k}")
                    if a is not None:
                        break
            if a is None:
                te, va, tr = split(k)
                mk = MASKS[WIN]
                m = build_any(SEED0 + 100 * k + 15 + sd)
                m.fit(X[tr] * mk, Yany[tr], validation_data=(X[va] * mk, Yany[va]),
                      epochs=EPOCHS, batch_size=128, verbose=0)
                a = m.predict(X[te] * mk, batch_size=512, verbose=0)
                run.save_arm(f"MIANY_{WIN}_s{sd}_f{k}", a)
                tf.keras.backend.clear_session(); done += 1
            assert_arm_shape(a, len(te), name=f"MIANY_{WIN}_s{sd}_f{k}")
            PANY[sd][te] = a.ravel()
        run.log(f"  MI-any 시드{sd} · AUROC {_auc(YANY, PANY[sd]):.4f} "
                f"(최댓값 점수 {_auc(YANY, P[WIN][sd].max(1)):.4f})")

run.log(f"\n총 {time.time()-t0:.0f}s · 이번 세션 학습 {done}회")

In [ ]:
# CELL 4 — 【P-1】 승자 6시드 확정 + 【P-4】 앙상블 이득 (양쪽 공정 비교)
from scipy import stats
from sklearn.metrics import average_precision_score

def t_ci(vals, conf=0.95):
    v = np.asarray([x for x in vals if np.isfinite(x)], float); n = len(v)
    m = float(v.mean()) if n else float("nan")
    if n < 2:
        return m, np.nan, np.nan, 0.0
    sd = float(v.std(ddof=1))
    h = float(stats.t.ppf(.5 + conf / 2, n - 1) * sd / np.sqrt(n))
    return m, m - h, m + h, sd

AU = {c: {s: {sd: float(_auc(Ymul[:, j].astype(bool), P[c][sd][:, j]))
              for sd in WANT[c]} for j, s in enumerate(SITES)} for c in WANT}
def Dv(c, s):
    sds = [x for x in WANT[c] if x in WANT[REF]]
    return [AU[REF][s][x] - AU[c][s][x] for x in sds]

run.log("\n" + "=" * 110)
run.log(f"【P-1】 {WIN}({N_ELEC[WIN]}전극) 6시드 확정 — D < {D_THR}")
run.log("=" * 110)
p1, worst = [], (None, -9)
for s in SITES:
    m, lo, hi, sd = t_ci(Dv(WIN, s))
    v = decide(lo, hi, D_THR, "<"); p1.append(v)
    if m > worst[1]:
        worst = (s, m)
    run.log(f"  {s:<7}{SITE_PLANE[s]:<7}D = {m:>+8.4f} [{lo:+.4f}, {hi:+.4f}] "
            f"(SD {sd:.4f}) → {MARK[v]}")
V = {}
V["P-1"] = True if all(x is True for x in p1) else \
    False if any(x is False for x in p1) else None
run.log(f"  P-1 → {MARK[V['P-1']]}  최악 {worst[0]} D={worst[1]:+.4f}")

run.log("\n" + "=" * 110)
run.log("【P-4】 앙상블 이득 — (시드평균 예측의 AUROC) − (시드별 AUROC 의 평균)")
run.log("=" * 110)
run.log("  ★ 실험18 A1 은 한쪽만 앙상블했다. 여기서는 **모든 구성에 똑같이** 건다.")
ENS, gains = {}, []
run.log(f"  {'구성':<13}" + "".join(f"{s:>9}" for s in SITES) + f"{'중앙값':>9}")
for c in WANT:
    sds = WANT[c][:3]                      # 공정 비교를 위해 앞 3시드로 통일
    row = {}
    for j, s in enumerate(SITES):
        y = Ymul[:, j].astype(bool)
        avg = float(_auc(y, np.mean([P[c][x][:, j] for x in sds], axis=0)))
        row[s] = avg - float(np.mean([AU[c][s][x] for x in sds]))
    ENS[c] = row; gains += list(row.values())
    run.log(f"  {c:<13}" + "".join(f"{row[s]:>+9.4f}" for s in SITES)
            + f"{np.median(list(row.values())):>+9.4f}")
m4, lo4, hi4, sd4 = t_ci([np.median(list(ENS[c].values())) for c in WANT])
V["P-4"] = decide(lo4, hi4, 0.0, ">")
run.log(f"\n  P-4 → 구성별 중앙값의 평균 {m4:+.4f} [{lo4:+.4f}, {hi4:+.4f}] → {MARK[V['P-4']]}")
run.log(f"      전체 {len(gains)}칸 중 양수 {sum(1 for g in gains if g > 0)}칸 · "
        f"전체 중앙값 {np.median(gains):+.4f}")

In [ ]:
# CELL 5 — 【P-5】 라우팅 재측정 (양쪽 앙상블 · V2 구성 포함)
run.log("\n" + "=" * 110)
run.log("【P-5】 부위별 라우팅 — 실험18 A1 재측정. **양쪽 다 앙상블**하고 후보에 V2 포함")
run.log("=" * 110)
POOL = ["I+II", "I+II+V1", "I+II+V2", "I+II+V1+V5", WIN]
SDS = SEEDS_OLD
run.log(f"  후보 {POOL} · 앙상블 시드 {SDS}")

def ens_pred(c, j, idx=None):
    """구성 c·부위 j 의 **시드평균 예측**. 라우팅과 고정 양쪽에 똑같이 쓴다."""
    v = np.mean([P[c][x][:, j] for x in SDS], axis=0)
    return v if idx is None else v[idx]

# 고정 구성들(앙상블)의 부위별 AUROC — 라우팅의 정직한 상대
FIX = {c: {s: float(_auc(Ymul[:, j].astype(bool), ens_pred(c, j)))
           for j, s in enumerate(SITES)} for c in POOL + [REF]}
route, RAU = {}, {}
for j, s in enumerate(SITES):
    y = Ymul[:, j].astype(bool)
    picked, sel = np.zeros(len(EID), "float32"), []
    for k in range(K_FOLD):
        te, other = np.where(CV == k)[0], np.where(CV != k)[0]
        best = max(POOL, key=lambda c: _auc(y[other], ens_pred(c, j, other)))
        sel.append(best); picked[te] = ens_pred(best, j, te)
    route[s] = sel
    RAU[s] = float(_auc(y, picked))

# 최선 고정 구성도 **중첩 선택**으로 고른다(라우팅과 같은 조건: 부위 무관 단일 구성)
fix_sel = []
for k in range(K_FOLD):
    other = np.where(CV != k)[0]
    fix_sel.append(max(POOL, key=lambda c: np.mean(
        [_auc(Ymul[other, j].astype(bool), ens_pred(c, j, other)) for j in range(NS)])))
BEST_FIX = max(set(fix_sel), key=fix_sel.count)
run.log(f"  최선 고정 구성(중첩 선택) = {BEST_FIX}  겹별 {fix_sel}")

run.log(f"\n  {'부위':<7}{'라우팅 선택':<14}{'라우팅 D':>10}{'고정 ' + BEST_FIX + ' D':>18}{'차':>9}")
dd = []
for s in SITES:
    dr = FIX[REF][s] - RAU[s]
    dfx = FIX[REF][s] - FIX[BEST_FIX][s]
    dd.append(dr - dfx)
    uniq = max(set(route[s]), key=route[s].count)
    n_uniq = len(set(route[s]))
    run.log(f"  {s:<7}{uniq + ('*' if n_uniq > 1 else ''):<14}{dr:>+10.4f}{dfx:>+18.4f}"
            f"{dr - dfx:>+9.4f}")
R_WORST = max(FIX[REF][s] - RAU[s] for s in SITES)
F_WORST = max(FIX[REF][s] - FIX[BEST_FIX][s] for s in SITES)
run.log(f"\n  최악 D — 라우팅 {R_WORST:+.4f} vs 고정 {BEST_FIX} {F_WORST:+.4f}")
V["P-5"] = bool(R_WORST < F_WORST - 0.005)
run.log(f"  P-5 → {MARK[V['P-5']]}  "
        + ("라우팅 이득 있음 → 실험20(평면 분리 헤드) 착수 근거"
           if V["P-5"] else "라우팅 이득 없음 → **실험20 은 만들 필요 없다**"))
run.log("      (* = 겹마다 선택이 갈린 부위. 전 겹 동일이면 라우팅 = 고정 배정이다)")

In [ ]:
# CELL 6 — 【P-2·P-3】 2단계 동작점 (이 실험의 본체)
run.log("\n" + "=" * 118)
run.log("【P-2·P-3】 2단계 동작점 — 1단계 MI유무 게이트 → 2단계 부위")
run.log("=" * 118)

def thr_at_sens(score, pos, target):
    """검증 집합에서 민감도 target 을 주는 임계값(교차적합용)."""
    p = score[pos]
    return float(np.quantile(p, 1.0 - target, method="lower")) if len(p) else -np.inf

def gate_thr(score, va, kind):
    """1단계 임계값.
      any/head — 그 점수에 MI-any 민감도 SENS_L1. 흔한 부위가 임계값을 정할 위험.
      safe     — **모든 부위**가 각각 SENS_L1 을 넘도록(부위별 분위수의 최솟값).
                 안전하지만 게이트가 아무것도 못 거를 수 있다.
    """
    if kind == "safe":
        return min(thr_at_sens(score[va], Ymul[va, j].astype(bool), SENS_L1)
                   for j in range(NS))
    return thr_at_sens(score[va], YANY[va], SENS_L1)

GATE_SCORE = {"any": lambda sd: P[WIN][sd].max(1),
              "safe": lambda sd: P[WIN][sd].max(1)}
if ANY_HEAD:
    GATE_SCORE["head"] = lambda sd: PANY[sd]          # ★ 전용 헤드 점수

def eval_design(c, sd, gate):
    """단일단계 vs 2단계를 **부위마다 도달 가능한 같은 민감도**에서 비교한다.

    ★ 두 단계는 독립이 아니다. 0.95 x 0.90 = 0.855 로 계산하면 틀린다 —
      1단계 게이트의 민감도는 MI-any 기준이라, 특정 부위 양성례가 그 게이트를
      통과할 확률(=그 부위의 도달 가능 최대 민감도)은 부위마다 다르다.
      픽스처에서 희소 부위 통과율이 0.67~0.78 까지 떨어졌다.
    → 부위별로 목표 민감도를 `min(SENS_FINAL, 게이트 통과율 x 0.98)` 로 낮춰 잡고,
      **단일단계도 같은 값으로** 평가한다. 도달 가능성 자체도 함께 보고한다.
    """
    s1 = GATE_SCORE[gate](sd)
    one, two, meta = {}, {}, {}
    for j, s in enumerate(SITES):
        y = Ymul[:, j].astype(bool)
        a1 = np.zeros(len(EID), bool); a2 = np.zeros(len(EID), bool)
        tgt_log, pass_log = [], []
        for k in range(K_FOLD):
            te, va, _ = split(k)
            t_l1 = gate_thr(s1, va, gate)
            pos = va[y[va]]
            if len(pos) == 0:
                continue
            passed = s1[pos] >= t_l1
            pr = float(passed.mean())                  # 이 부위의 도달 가능 최대 민감도
            tgt = min(SENS_FINAL, pr * 0.98)           # 두 설계에 **똑같이** 적용
            pass_log.append(pr); tgt_log.append(tgt)
            # 단일단계 — 같은 목표 민감도
            a1[te] = P[c][sd][te, j] >= thr_at_sens(P[c][sd][va, j], y[va], tgt)
            # 2단계 — 게이트 탈락자는 놓친 것으로 세고 목표를 맞춘다
            sc = np.where(passed, P[c][sd][pos, j], -np.inf)
            t_l2 = float(np.quantile(sc, 1.0 - tgt, method="lower"))
            a2[te] = (s1[te] >= t_l1) & (P[c][sd][te, j] >= t_l2)
        for tag, a in (("one", a1), ("two", a2)):
            tp = int((a & y).sum()); n_a = int(a.sum())
            (one if tag == "one" else two)[s] = {
                "sens": tp / max(int(y.sum()), 1), "alarm": n_a / len(EID),
                "ppv": tp / max(n_a, 1)}
        meta[s] = {"gate_pass": float(np.mean(pass_log)) if pass_log else np.nan,
                   "target": float(np.mean(tgt_log)) if tgt_log else np.nan}
    return one, two, meta

# ── 두 게이트를 모두 재고 나은 쪽을 본다
RES = {}
GATES = ("any", "safe") + (("head",) if ANY_HEAD else ())
for gate in GATES:
    O, T, MT = {}, {}, None
    for sd in SEEDS_OLD:
        O[sd], T[sd], MT = eval_design(WIN, sd, gate)
    RES[gate] = (O, T, MT)
    gp = np.mean([MT[s]["gate_pass"] for s in SITES])
    run.log(f"\n  게이트 '{gate}' — 부위 양성례의 게이트 통과율 평균 {gp:.3f}")
    run.log("      " + " ".join(f"{s} {MT[s]['gate_pass']:.2f}" for s in SITES))

# 사전지정: 희소 4부위에서 도달 가능 민감도가 높은 게이트를 쓴다
#   ①희소 4부위 통과율이 목표 이상이고 ②경보를 가장 많이 줄이는 게이트를 쓴다.
#   통과율이 목표에 못 미치면 그 게이트는 민감도를 깎으므로 후보에서 뺀다.
def _ok(g):
    return all(RES[g][2][s]["gate_pass"] >= SENS_FINAL for s in SCARCE)
def _cut(g):
    O_, T_, _ = RES[g]
    return float(np.median([np.mean([T_[sd][s]["alarm"] - O_[sd][s]["alarm"]
                                     for sd in SEEDS_OLD]) for s in SCARCE]))
CAND = [g for g in GATES if _ok(g)] or list(GATES)
GATE = min(CAND, key=_cut)
run.log(f"\n  게이트별 희소 4부위 경보 변화: "
        + " · ".join(f"{g} {_cut(g):+.4f}{'' if _ok(g) else '(민감도 미달)'}" for g in GATES))
ONE, TWO, META = RES[GATE]
run.log(f"  → 채택 게이트: '{GATE}'")

run.log(f"\n  {'부위':<7}{'유병률':>8}{'게이트통과':>10}{'목표민감도':>11}"
        f"{'단일 경보':>10}{'단일 PPV':>10}{'2단계 경보':>11}{'2단계 PPV':>11}{'경보변화':>10}")
prev = {s: float(Ymul[:, j].mean()) for j, s in enumerate(SITES)}
for s in SITES:
    o = {k: np.mean([ONE[sd][s][k] for sd in SEEDS_OLD]) for k in ("sens", "alarm", "ppv")}
    t = {k: np.mean([TWO[sd][s][k] for sd in SEEDS_OLD]) for k in ("sens", "alarm", "ppv")}
    run.log(f"  {s:<7}{prev[s]:>8.4f}{META[s]['gate_pass']:>10.3f}{META[s]['target']:>11.3f}"
            f"{o['alarm']:>10.3f}{o['ppv']:>10.1%}{t['alarm']:>11.3f}{t['ppv']:>11.1%}"
            f"{1 - t['alarm'] / max(o['alarm'], 1e-9):>9.0%}")

# 전제 확인 — 두 설계의 실측 민감도가 같아야 경보율 비교가 성립한다
so = np.mean([np.mean([ONE[sd][s]["sens"] for s in SITES]) for sd in SEEDS_OLD])
sv = np.mean([np.mean([TWO[sd][s]["sens"] for s in SITES]) for sd in SEEDS_OLD])
run.log(f"\n  【전제 확인】 실측 민감도 평균 — 단일 {so:.3f} · 2단계 {sv:.3f}")
SENS_MATCHED = abs(so - sv) <= 0.03
run.log("  " + ("✅ 민감도가 맞춰졌다 — 경보율·PPV 비교가 성립한다" if SENS_MATCHED else
                "⛔ 민감도가 0.03 넘게 벌어졌다 — **아래 P-2·P-3 을 그대로 읽으면 안 된다**"))

d2 = [float(np.median([TWO[sd][s]["alarm"] - ONE[sd][s]["alarm"] for s in SCARCE]))
      for sd in SEEDS_OLD]
m2, lo2, hi2, sd2 = t_ci(d2)
V["P-2"] = decide(lo2, hi2, 0.0, "<") if SENS_MATCHED else None
run.log(f"\n  P-2 희소 4부위 {SCARCE} 경보율 변화 중앙값 = {m2:+.4f} "
        f"[{lo2:+.4f}, {hi2:+.4f}] → {MARK[V['P-2']]}")

r3 = [float(np.median([TWO[sd][s]["ppv"] / max(ONE[sd][s]["ppv"], 1e-9) for s in SCARCE]))
      for sd in SEEDS_OLD]
m3, lo3, hi3, sd3 = t_ci(r3)
V["P-3"] = decide(lo3, hi3, PPV_MULT, ">") if SENS_MATCHED else None
run.log(f"  P-3 희소 4부위 PPV 배수 중앙값 = {m3:.2f}배 [{lo3:.2f}, {hi3:.2f}] "
        f"vs {PPV_MULT} → {MARK[V['P-3']]}")

# 게이트가 왜 손해인지/이득인지 진단 — 다음 실험의 입력
run.log("\n  【진단】 1단계 게이트가 희소 부위 양성례를 떨어뜨리는가")
low = [s for s in SITES if META[s]["gate_pass"] < SENS_FINAL]
if low:
    run.log(f"      ⚠️ 게이트만으로 목표({SENS_FINAL:.3f})에 못 미치는 부위 {low}")
    run.log("      → 1단계 점수를 '부위 확률의 최댓값' 으로 쓴 것이 원인이다. 그 점수는")
    run.log("        흔한 부위가 임계값을 정해서 희소·어려운 부위 양성례를 먼저 버린다.")
    run.log("        → 다음: **MI-any 전용 헤드를 따로 학습**하거나 부위별 게이트를 쓴다")
else:
    run.log("      ✅ 모든 부위가 게이트를 목표 이상으로 통과한다 — 게이트가 안전하다")

run.log("\n" + "=" * 118)
for k in ("P-1", "P-2", "P-3", "P-4", "P-5"):
    run.log(f"  {k}: {MARK[V.get(k)]}")
run.log("=" * 118)

In [ ]:
# CELL 7 — 그림
import matplotlib.pyplot as plt
xs = np.arange(len(SITES)); w = 0.38
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.8))

ax[0].bar(xs, [np.mean(Dv(WIN, s)) for s in SITES], .55,
          yerr=[t_ci(Dv(WIN, s))[3] for s in SITES], capsize=3, color="tab:green")
ax[0].axhline(0, color="k", lw=1); ax[0].axhline(D_THR, color="red", ls="--", lw=1.3)
ax[0].text(-.4, D_THR + .001, f"관문 {D_THR}", color="red", fontsize=8)
ax[0].set_xticks(xs); ax[0].set_xticklabels(SITES, rotation=45, ha="right")
ax[0].set_ylabel("잔여 결손 D")
ax[0].set_title(f"{WIN} ({N_ELEC[WIN]}전극) 6시드 · P-1 {MARK[V['P-1']]}")

o_al = [np.mean([ONE[sd][s]["alarm"] for sd in SEEDS_OLD]) for s in SITES]
t_al = [np.mean([TWO[sd][s]["alarm"] for sd in SEEDS_OLD]) for s in SITES]
ax[1].bar(xs - w/2, o_al, w, label="단일단계", color="0.6")
ax[1].bar(xs + w/2, t_al, w, label="2단계", color="tab:orange")
ax[1].set_xticks(xs); ax[1].set_xticklabels(SITES, rotation=45, ha="right")
ax[1].set_ylabel("경보율"); ax[1].legend(fontsize=8)
ax[1].set_title(f"경보 부담 · P-2 {MARK[V['P-2']]}")

o_pv = [np.mean([ONE[sd][s]["ppv"] for sd in SEEDS_OLD]) for s in SITES]
t_pv = [np.mean([TWO[sd][s]["ppv"] for sd in SEEDS_OLD]) for s in SITES]
ax[2].bar(xs - w/2, o_pv, w, label="단일단계", color="0.6")
ax[2].bar(xs + w/2, t_pv, w, label="2단계", color="tab:blue")
ax[2].set_xticks(xs); ax[2].set_xticklabels(SITES, rotation=45, ha="right")
ax[2].set_ylabel("PPV"); ax[2].legend(fontsize=8)
ax[2].set_title(f"경보 1건이 진짜일 확률 · P-3 {MARK[V['P-3']]}")

plt.tight_layout(); run.save_fig("two_stage", fig); plt.show()

In [ ]:
# CELL 8 — 결과 저장
res = {
    "week": 2, "exp_id": "exp19_two_stage", "quest": "ailab-2026-0015",
    "task": "동작점 2단계 재설계 — 유도는 천장에 닿았고 남은 병목은 유병률이다",
    "split": "inter", "step": "exp19-two-stage-operating-point",
    "metric": "scarce_ppv_ratio_two_vs_one", "value": round(float(m3), 3),
    "passed": bool(V.get("P-2") is True and V.get("P-3") is True),
    "date": time.strftime("%Y-%m-%d"),
    "k_fold": K_FOLD, "n_seeds": {c: len(WANT[c]) for c in WANT},
    "sites": SITES, "scarce": SCARCE, "winner": WIN, "n_elec": N_ELEC[WIN],
    "reuse_ok": bool(REUSE_OK), "drift_max_abs": round(float(DRIFT_MAX), 4),
    "trained_this_run": int(done), "sec_per_training": round(float(PER), 1),
    "gate_used": GATE, "sens_matched": bool(SENS_MATCHED),
    "gate_pass_by_site": {s: round(float(META[s]["gate_pass"]), 4) for s in SITES},
    "target_sens_by_site": {s: round(float(META[s]["target"]), 4) for s in SITES},
    "sites_below_target_by_gate": [s for s in SITES
                                   if META[s]["gate_pass"] < SENS_FINAL],
    "two_stage": {"sens_l1": SENS_L1, "sens_final_target": round(SENS_FINAL, 4),
                  "sens_measured_one": round(float(so), 4),
                  "sens_measured_two": round(float(sv), 4)},
    "P-1": V.get("P-1"), "P-2": V.get("P-2"), "P-3": V.get("P-3"),
    "P-4": V.get("P-4"), "P-5": V.get("P-5"),
    "p1_worst_site": worst[0], "p1_worst_D": round(float(worst[1]), 4),
    "p2_alarm_delta_median": round(float(m2), 4),
    "p2_ci": [round(float(lo2), 4), round(float(hi2), 4)],
    "p3_ppv_ratio": round(float(m3), 3),
    "p3_ci": [round(float(lo3), 3), round(float(hi3), 3)],
    "p4_ensemble_gain": round(float(m4), 4),
    "p4_ci": [round(float(lo4), 4), round(float(hi4), 4)],
    "p4_gain_by_config": {c: round(float(np.median(list(ENS[c].values()))), 4) for c in ENS},
    "p5_routing_worst_D": round(float(R_WORST), 4),
    "p5_fixed_worst_D": round(float(F_WORST), 4),
    "p5_best_fixed": BEST_FIX,
    "p5_route": {s: max(set(route[s]), key=route[s].count) for s in SITES},
    "p5_route_varied": [s for s in SITES if len(set(route[s])) > 1],
    "D_mean": {c: {s: round(float(np.mean(Dv(c, s))), 4) for s in SITES} for c in WANT},
    "one_stage": {s: {k: round(float(np.mean([ONE[sd][s][k] for sd in SEEDS_OLD])), 4)
                      for k in ("sens", "alarm", "ppv")} for s in SITES},
    "two_stage_result": {s: {k: round(float(np.mean([TWO[sd][s][k] for sd in SEEDS_OLD])), 4)
                             for k in ("sens", "alarm", "ppv")} for s in SITES},
    "verdict": " · ".join(f"{k} {MARK[V.get(k)]}" for k in
                          ("P-1", "P-2", "P-3", "P-4", "P-5")),
}
res["summary"] = (f"{WIN} 6시드 최악 D {res['p1_worst_D']:+.4f} · "
                  f"희소 PPV {m3:.2f}배 · 경보 {m2:+.4f} · "
                  f"앙상블 {m4:+.4f} · 라우팅 {R_WORST:+.4f} vs 고정 {F_WORST:+.4f} · "
                  + res["verdict"])
run.save_json("result.json", res)
run.log("\n" + json.dumps({k: res[k] for k in
                           ("metric", "value", "passed", "P-1", "P-2", "P-3", "P-4",
                            "P-5", "summary")}, ensure_ascii=False, indent=2))
run.finish(res)